# Week 10: Multimodal LLMs

Every model so far has read only text. This week we give our models eyes, in two steps.

- **Part A (Class 1): a shared space for images and text.** We see how CLIP learns to put a photo and its caption at the same spot, reuse Week 9's `build_index()` / `search()` to find photos from a sentence, and find a simple question CLIP cannot answer.
- **Part B (Class 2): a model that talks about images.** We plug a vision encoder into a chat model like the Qwen we used in Weeks 7 to 9, look at the prompt it actually reads, and help AeroBeat's customer Sara, who sends us a photo of her receipt.

Run on a Colab T4 (Runtime → Change runtime type → T4 GPU).

# Part A (Class 1): Images and Text in One Space

## A.0 Setup

Our model for Part A is CLIP (`openai/clip-vit-base-patch32`). For photos we use **Imagenette**, ten easy classes of real ImageNet photos. We take the first 3 photos of each class, a 30-photo library: the image version of Week 9's help center.

In [ ]:
import os, tarfile, urllib.request
import numpy as np, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

if not os.path.isdir("imagenette2-160/val"):                         # download once, 94 MB
    urllib.request.urlretrieve("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", "imagenette.tgz")
    tarfile.open("imagenette.tgz").extractall(filter="data")

NAMES = {"n01440764": "tench", "n02102040": "English springer", "n02979186": "cassette player",
         "n03000684": "chain saw", "n03028079": "church", "n03394916": "French horn",
         "n03417042": "garbage truck", "n03425413": "gas pump", "n03445777": "golf ball",
         "n03888605": "parachute"}                                   # ImageNet folder name -> class name
folders = sorted(os.listdir("imagenette2-160/val"))
print("folders on disk:", folders)
CLASSES = {folder: NAMES.get(folder, folder) for folder in folders}  # unknown folder: keep its own name
LIBRARY = [(f"imagenette2-160/val/{folder}/{f}", name) for folder, name in CLASSES.items()
           for f in sorted(os.listdir(f"imagenette2-160/val/{folder}"))[:3]]
photos = [Image.open(path).convert("RGB") for path, _ in LIBRARY]
labels = [name for _, name in LIBRARY]

def show(images, titles, ncols, size=1.6):                           # small helper to draw a grid of images
    nrows = (len(images) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows + 0.3 * nrows))
    for ax in np.array(axes).flat: ax.axis("off")
    for ax, img, t in zip(np.array(axes).flat, images, titles):
        ax.imshow(img); ax.set_title(t, fontsize=8)
    plt.tight_layout(); plt.show()

show(photos, labels, ncols=10)

## A.1 How CLIP reads an image and a sentence

CLIP has two encoders. The text encoder is a transformer like Week 5's. The image encoder is a **Vision Transformer (ViT)**: it cuts the image into square patches and treats each patch as one token, the way Week 2 cut a sentence into tokens. CLIP ViT-B/32 cuts a 224 × 224 image into 32 × 32 patches, so 7 × 7 = 49 "image words". (How vision encoders work inside is covered in the Computer Vision course; this one idea is all we need.)

Each encoder ends with one summary vector, projected into a **shared 512-dim space**. A photo of a dog and the caption "a dog" should land near each other there. Nothing in the architecture forces that; training does.

## A.2 How CLIP learns the shared space

CLIP was trained on 400 million (photo, caption) pairs from the web. Each training step takes a batch of pairs and plays a quiz: every photo must pick its own caption out of all the captions in the batch, and every caption must pick its own photo. The other pairs' captions serve as free wrong answers. This is called **contrastive learning**, and Week 13 uses it again.

We follow one tiny batch of two pairs: (dog photo, "a dog") and (car photo, "a car").

**Step 1: the four vectors.** Each vector is [dog-like, car-like]. These are hand-picked toy values; real CLIP vectors have 512 numbers.

| Item | Vector |
|---|---|
| caption "a dog" | [1, 0] |
| caption "a car" | [0, 1] |
| dog photo, before training | [0.28, 0.96] |
| car photo, before training | [0.96, 0.28] |

Every vector has length 1, for example $\sqrt{0.28^2 + 0.96^2} = \sqrt{0.0784 + 0.9216} = 1$. So cosine is just the dot product (Week 9).

**Step 2: before training, the dog photo is far from its caption.** The untrained image encoder put the dog photo on the car side.

- dog photo vs "a dog": $0.28 \times 1 + 0.96 \times 0 = 0.28$
- dog photo vs "a car": $0.28 \times 0 + 0.96 \times 1 = 0.96$ (closest, but wrong)

The car photo is the mirror image: 0.96 with "a dog" (wrong) and 0.28 with "a car".

**Step 3: before training, the loss.** Quiz: "Dog photo, which caption is yours?" Softmax turns the two cosines into probabilities (Week 4):

$$e^{0.28} = 1.323, \quad e^{0.96} = 2.612, \quad \text{total} = 3.935, \quad p(\text{"a dog"}) = 1.323 / 3.935 = 0.336$$

The loss is the cross-entropy from Weeks 4 and 5:

$$\text{loss} = -\ln 0.336 = 1.090$$

The model is 66% sure the dog photo shows a car, so the loss is high.

**Step 4: training moves the photos.** Backpropagation (Week 1) uses this loss to adjust the image encoder's weights, so that each photo comes out a little closer to its own caption and farther from the other, round after round. After training, the encoder gives (hand-picked to show the result):

| Item | Before | After |
|---|---|---|
| caption "a dog" | [1, 0] | [1, 0] |
| caption "a car" | [0, 1] | [0, 1] |
| dog photo | [0.28, 0.96] | [0.8, 0.6] |
| car photo | [0.96, 0.28] | [0.6, 0.8] |

To keep the picture simple, only the photos move here. In real CLIP both encoders train, so the captions move too.

**Step 5: after training, the dog photo is close to its caption.**

- dog photo vs "a dog": $0.8 \times 1 + 0.6 \times 0 = 0.80$ (was 0.28)
- dog photo vs "a car": $0.8 \times 0 + 0.6 \times 1 = 0.60$ (was 0.96)

The car photo again mirrors it: 0.60 with "a dog" and 0.80 with "a car". Each photo is now closest to its own caption, without sitting exactly on it.

**Step 6: after training, the loss.**

$$e^{0.80} = 2.226, \quad e^{0.60} = 1.822, \quad \text{total} = 4.048, \quad p(\text{"a dog"}) = 2.226 / 4.048 = 0.550$$

$$\text{loss} = -\ln 0.550 = 0.598$$

| Stage | dog photo | cos with "a dog" | cos with "a car" | p("a dog") | Loss |
|---|---|---|---|---|---|
| before training | [0.28, 0.96] | 0.28 | 0.96 | 0.336 | 1.090 |
| after training | [0.8, 0.6] | 0.80 | 0.60 | 0.550 | 0.598 |

55% may look unsure. That is because cosines only range from $-1$ to $1$. Real CLIP multiplies every cosine by a learned scale of 100 before the softmax, so 0.80 against 0.60 becomes 80 against 60, and the right caption wins with essentially 100%. You will see this scale as `clip.logit_scale` in A.5.

## A.3 Searching photos with a sentence (Week 9, new encoders)

Week 9's search had two steps: embed every document once (`build_index`), then embed the query and rank by cosine (`search`). Neither step cares what a "document" is. Use CLIP's image encoder for the index and its text encoder for the query, and the same two functions search photos.

In [ ]:
def embed_images(images):
    with torch.no_grad():
        v = clip.get_image_features(**clip_proc(images=images, return_tensors="pt").to(device))
    v = v if torch.is_tensor(v) else v.pooler_output     # newer transformers wrap the vector
    return F.normalize(v, dim=-1)

def embed_texts(texts):
    with torch.no_grad():
        v = clip.get_text_features(**clip_proc(text=texts, return_tensors="pt", padding=True).to(device))
    v = v if torch.is_tensor(v) else v.pooler_output
    return F.normalize(v, dim=-1)

def build_index(images):                  # Week 9: build_index(chunks) with MiniLM
    return embed_images(images)

def search(query, index, k=3):            # Week 9: rank by cosine, keep the top k
    top = (embed_texts([query]) @ index.T)[0].topk(k)
    return [(s.item(), i.item()) for s, i in zip(top.values, top.indices)]

index = build_index(photos)
QUERIES = ["a man holding a big fish", "something you play music with",
           "where you fill up your car", "jumping out of a plane"]
imgs, titles = [], []
for q in QUERIES:
    for score, i in search(q, index):
        imgs.append(photos[i]); titles.append(f"{score:.3f} {labels[i]}")
    print(q, "->", titles[-3:])
show(imgs, titles, ncols=3, size=1.8)

The library is only pixels: no captions, labels or file names were used. "Where you fill up your car" shares no word with "gas pump", just as Week 9's "send them back" shared no word with "Returns".

The scores sit around 0.2 to 0.35 even for good matches, because image and text vectors never line up as closely as two sentences do. As in Week 9, the rank matters, not the raw number.

## A.4 Zero-shot classification

A classifier is just a search where the "documents" are the class names. Embed one caption per class and give each photo the closest one. Nothing is trained; that is **zero-shot**. Week 5 needed thousands of labelled reviews and a training loop to reach 80%.

In [ ]:
names = list(CLASSES.values())
for template in ["{}", "a photo of a {}."]:
    text = embed_texts([template.format(n) for n in names])
    preds = (index @ text.T).argmax(1).tolist()
    correct = sum(names[p] == y for p, y in zip(preds, labels))
    print(f"{template!r:20} {correct}/{len(labels)} correct")

Ten easy classes are not much of a test, so both prompts do well. On full ImageNet (1,000 classes), the CLIP paper reports that "a photo of a {}." beats the bare label by about 1.3 points, because CLIP's training captions were sentences. It is Week 7's lesson again: prompt wording matters, even for images.

## A.5 Where CLIP fails: who is where

CLIP gives one vector per image. Here is a simple scene and three captions that use almost the same words.

In [ ]:
def two_shapes(left, right):                         # each is (shape, color)
    img = Image.new("RGB", (224, 224), "white")
    d = ImageDraw.Draw(img)
    for (shape, color), x0 in [(left, 20), (right, 124)]:
        box = (x0, 72, x0 + 80, 152)
        (d.rectangle if shape == "square" else d.ellipse)(box, fill=color)
    return img

scene = two_shapes(("square", "red"), ("circle", "blue"))
CAPTIONS = ["a red square to the left of a blue circle",   # true
            "a blue circle to the left of a red square",   # same words, order flipped
            "a blue square to the left of a red circle"]   # colors swapped
with torch.no_grad():
    probs = (clip.logit_scale.exp() * embed_images([scene]) @ embed_texts(CAPTIONS).T).softmax(-1)[0]
show([scene], [""], ncols=1, size=2.2)
for c, p in zip(CAPTIONS, probs): print(f"{p:.2f}  {c}")

If the probabilities split between the true caption and the flipped one, CLIP cannot tell left from right. It is not a bug in our code. During training, a batch almost never held two captions with the same words in a different order, so CLIP only needed to learn **which things are present**, not **how they relate**. It behaves much like a bag of words.

To answer "what is on the left?", a model must keep all the patch tokens and let words attend to them while it writes. That is a vision-language model, and it is next class.

# Part B (Class 2): A Language Model That Sees

## B.0 Three parts of a vision-language model

Every vision-language model (VLM) in the slides has three parts:

1. **Vision encoder:** a ViT, as in Class 1, gives one vector per patch.
2. **Connector:** turns those vectors into "soft visual tokens" the size the LLM expects. BLIP-2 uses the Q-Former: 32 learned queries attend to the patches, so every image becomes exactly 32 tokens. Our model uses a simpler connector: it merges each 2 × 2 block of neighbor patches and passes it through a small MLP.
3. **LLM:** an ordinary chat model that reads image tokens and text tokens in one prompt.

Our model is **Qwen2-VL-2B-Instruct**: a 675M-parameter ViT plus the 1.5B Qwen2 language model, the same family as the Qwen2.5-1.5B we used in Weeks 7 to 9. In fp16 it fits on the T4.

### Setup

`ask()` plays the role of Week 8's `generate()`: build the chat messages, generate greedily, return only the new text. The only change is that a message can now hold an image.

In [ ]:
# If the runtime restarted, run A.0, A.3 and A.5 first: Part B reuses photos, labels and scene.
import re
from datetime import date
from PIL import ImageFont
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

VLM_ID = "Qwen/Qwen2-VL-2B-Instruct"
vlm = Qwen2VLForConditionalGeneration.from_pretrained(VLM_ID, torch_dtype=torch.float16, device_map="cuda").eval()
vlm_proc = AutoProcessor.from_pretrained(VLM_ID)
IMAGE_PAD = vlm_proc.tokenizer.convert_tokens_to_ids("<|image_pad|>")

visual = getattr(vlm, "visual", None) or vlm.model.visual
n_vit = sum(p.numel() for p in visual.parameters())
print(f"weights in fp16: {vlm.get_memory_footprint() / 1e9:.1f} GB")
print(f"vision encoder: {n_vit / 1e6:.0f}M parameters, everything else: {(vlm.num_parameters() - n_vit) / 1e6:.0f}M")

def build(messages, image=None):
    text = vlm_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = vlm_proc(text=[text], images=[image] if image is not None else None, return_tensors="pt").to("cuda")
    return text, inputs

def user_message(question, image=None):
    content = ([{"type": "image"}] if image is not None else []) + [{"type": "text", "text": question}]
    return {"role": "user", "content": content}

def ask(question, image=None, system=None, max_new_tokens=120):
    messages = ([{"role": "system", "content": system}] if system else []) + [user_message(question, image)]
    _, inputs = build(messages, image)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return vlm_proc.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

If the answers come out empty or as repeated symbols, change `torch.float16` to `torch.float32` (about 9 GB, still fits on the T4).

## B.1 Look inside: an image is just more prompt

Week 7 printed the chat template string the model actually reads. Let's do the same with an image in the message, then count how many tokens the image becomes.

CLIP used 32 × 32 patches. Qwen2-VL's ViT uses smaller 14 × 14 patches, which keeps more detail, and the connector merges each 2 × 2 block into one token:

- 448 × 448 image: 448 / 14 = 32 patches per side, 32 × 32 = 1,024 patches, 1,024 / 4 = **256 image tokens**.
- 224 × 224 image: 224 / 14 = 16, 16 × 16 = 256 patches, 256 / 4 = **64 image tokens**.

In [ ]:
def image_tokens(image, question="Describe this photo in one sentence."):
    text, inputs = build([user_message(question, image)], image)
    _, h, w = inputs["image_grid_thw"][0].tolist()
    n = (inputs.input_ids == IMAGE_PAD).sum().item()
    return text, h, w, n, inputs.input_ids.shape[1]

for size in [448, 224]:
    text, h, w, n, total = image_tokens(photos[0].resize((size, size)))
    print(f"{size}x{size}: {h}x{w} = {h * w} patches -> {n} image tokens, {total} prompt tokens in all")
print("\nTemplate before the processor expands the image:\n" + text)

The template holds one `<|image_pad|>` between `<|vision_start|>` and `<|vision_end|>`. The processor copies it once per image token (64 or 256 times). Inside the model, the embeddings of those copies are replaced by the connector's outputs, the soft visual tokens from the slides.

So, as Week 8 put it, everything is a way of building the prompt. An image is a few hundred extra positions, and self-attention (Week 3) lets every word attend to every image token. That is exactly what CLIP's single vector could not do.

The trade-off differs from BLIP-2. The Q-Former always gives 32 tokens, cheap but fixed. Qwen2-VL gives more tokens to bigger images: more detail, more cost, like pasting longer chunks in Week 9.

## B.2 Describing a photo, and A.5's test again

In [ ]:
show([photos[0], scene], [labels[0], "A.5 scene"], ncols=2, size=2.2)
print("Photo:", ask("Describe this photo in one sentence.", photos[0].resize((448, 448))))
print("Left: ", ask("Which shape is on the left, and what color is it?", scene))
print("Check:", ask("Is this caption correct: 'a blue circle to the left of a red square'? Answer yes or no, then explain.", scene))

The description is free text, not a choice from a fixed list, and the shape answers depend on where things are. Generation over patch tokens handles the relation question that defeated CLIP.

## B.3 AeroBeat: Sara sends a photo of her receipt

Sara writes on October 28: "Can I still return these?" and attaches a photo of her receipt. We draw the receipt with PIL so we know exactly what it says. Like Week 9's help center, it is made up on purpose.

First the naive version: give the model the photo and the question.

In [ ]:
def receipt(bought):
    img = Image.new("RGB", (448, 448), "white")
    d = ImageDraw.Draw(img)
    big, font = ImageFont.load_default(size=30), ImageFont.load_default(size=20)
    d.text((30, 25), "AeroBeat Store", font=big, fill="black")
    lines = [("Order A-20417", ""), (f"Date: {bought}", ""), ("-" * 34, ""), ("AeroBeat Pro 2", ""),
             ("Wireless headphones", "$149.00"), ("Tax", "$8.94"), ("Total", "$157.94"),
             ("-" * 34, ""), ("Thank you, Sara!", "")]
    for i, (left, right) in enumerate(lines):
        d.text((30, 90 + 34 * i), left, font=font, fill="black")
        d.text((320, 90 + 34 * i), right, font=font, fill="black")
    return img

TODAY = date(2026, 10, 28)
RECEIPT = receipt("2026-09-20")
show([RECEIPT], [""], ncols=1, size=3)
print(ask(f"Today is {TODAY}. Can I still return these headphones?", RECEIPT))

Seeing the receipt does not tell the model AeroBeat's policy, so it guesses one (the usual guess is 30 days; AeroBeat's is 45). It may also get the day count wrong, since small models are poor at arithmetic.

The fix is a Week 8 chain with a Week 9 RAG step, where each part does what it is good at:

1. the VLM **reads** the date, in a fixed format we can parse in code,
2. Python **counts** the days,
3. the Returns article supplies the **policy**, in Week 9's prompt (question first, sources, soft way out last).

In the full system, Week 9's `search()` would pick the Returns article. Here we paste it directly, since retrieval is not new this week.

In [ ]:
SUPPORT = "You are a friendly customer support assistant for AeroBeat, an online headphone shop."   # Week 9
RETURNS = "You can return any AeroBeat product within 45 days of purchase for a full refund."      # Week 9 Returns article

raw = ask("What is the purchase date on this receipt? Reply with the date only, as YYYY-MM-DD.", RECEIPT, max_new_tokens=15)
bought = date.fromisoformat(re.search(r"\d{4}-\d{2}-\d{2}", raw).group())          # step 1: read
days = (TODAY - bought).days                                                       # step 2: count
question = f"I bought AeroBeat Pro 2 headphones {days} days ago. Can I still return them?"
prompt = (f"Question: {question}\n\nSources:\n[1] {RETURNS}\n\n"
          "Answer the question using the sources above, and cite the source number, like [1]. "
          "If none of the sources answers the question, say that you don't know.")
print("Step 1, read: ", raw)
print("Step 2, count:", days, "days")
print("Step 3, answer:", ask(prompt, system=SUPPORT))                                # step 3: RAG, text only

Three calls, and none of them asks a small model to do something it is bad at. Step 3 has no image at all: once the date is text, the VLM is just a text LLM with a Week 9 prompt.


## B.4 What can still go wrong

| Failure | Where we saw it | Fix | Week |
|---|---|---|---|
| CLIP matches words, not relations | A.5 | a VLM that attends to every patch | 10 |
| small text unreadable at low resolution | Try it | more pixels (more tokens) | 10 |
| policy invented from a photo | B.3 naive | RAG | 9 |
| describes objects that are not there | try your own photos | fine-tune on grounded data | 12 |
| text inside an image read as an instruction | | test our own systems | 14 |
| web captions carry stereotypes into CLIP | | test our own systems | 14 |

**Try it:** upload a photo of your own (Files panel on the left) and ask about it:

```python
print(ask("What is in this photo? List the objects.", Image.open("my_photo.jpg").convert("RGB")))
```

Shrink Sara's receipt to 112 × 112 (16 image tokens instead of 256, as B.1's arithmetic predicts) and ask for the date again. Does it still read it, or does it invent one?

```python
print(ask("What is the purchase date on this receipt?", RECEIPT.resize((112, 112))))
```

Ask about something that is **not** in the photo, for example "What color is the cat?" on a photo with no cat, and see whether the model admits it.